# Ferramentas

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier   # Analiza o Modelo
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix  # # Analiza o Modelo


import pandas as pd

# Importação dos dados

In [13]:
# Carregar o dataset
df = pd.read_csv('/content/survey lung cancer.csv')

# Visualizar as primeiras linhas
print(df.head())

  GENDER  AGE  SMOKING  YELLOW_FINGERS  ANXIETY  PEER_PRESSURE  \
0      M   69        1               2        2              1   
1      M   74        2               1        1              1   
2      F   59        1               1        1              2   
3      M   63        2               2        2              1   
4      F   63        1               2        1              1   

   CHRONIC DISEASE  FATIGUE   ALLERGY   WHEEZING  ALCOHOL CONSUMING  COUGHING  \
0                1         2         1         2                  2         2   
1                2         2         2         1                  1         1   
2                1         2         1         2                  1         2   
3                1         1         1         1                  2         1   
4                1         1         1         2                  1         2   

   SHORTNESS OF BREATH  SWALLOWING DIFFICULTY  CHEST PAIN LUNG_CANCER  
0                    2                      

# Pré-processamento dos dados

In [14]:
# Verificar tipos e dados ausentes
print("Tipos de dados:")
print(df.dtypes)

print("\nValores ausentes por coluna:")
print(df.isnull().sum())

# Ver valores únicos para entender as categorias
print("\nValores únicos por coluna:")
for col in df.columns:
    print(f"{col}: {df[col].unique()}")


Tipos de dados:
GENDER                   object
AGE                       int64
SMOKING                   int64
YELLOW_FINGERS            int64
ANXIETY                   int64
PEER_PRESSURE             int64
CHRONIC DISEASE           int64
FATIGUE                   int64
ALLERGY                   int64
WHEEZING                  int64
ALCOHOL CONSUMING         int64
COUGHING                  int64
SHORTNESS OF BREATH       int64
SWALLOWING DIFFICULTY     int64
CHEST PAIN                int64
LUNG_CANCER              object
dtype: object

Valores ausentes por coluna:
GENDER                   0
AGE                      0
SMOKING                  0
YELLOW_FINGERS           0
ANXIETY                  0
PEER_PRESSURE            0
CHRONIC DISEASE          0
FATIGUE                  0
ALLERGY                  0
WHEEZING                 0
ALCOHOL CONSUMING        0
COUGHING                 0
SHORTNESS OF BREATH      0
SWALLOWING DIFFICULTY    0
CHEST PAIN               0
LUNG_CANCER            

**GENDER** e **LUNG_CANCER** são do tipo object (texto) → precisam ser convertidos para números.

As demais variáveis **já são numéricas**, mas usam **1** e **2**, onde normalmente usamos 0 e 1 em ML.

Precisamos converter las

## **Etapas:**

*   Codificar GENDER → M=1, F=0
*   Codificar LUNG_CANCER → YES=1, NO=0
*   Corrigir as variáveis 1/2 → transformar em 0/1 para o modelo aprender melhor



In [15]:
# Criar cópia do DataFrame
df_encoded = df.copy()

# Codificar 'GENDER' (M → 1, F → 0)
df_encoded['GENDER'] = df_encoded['GENDER'].map({'M': 1, 'F': 0})

# Codificar 'LUNG_CANCER' (YES → 1, NO → 0)
df_encoded['LUNG_CANCER'] = df_encoded['LUNG_CANCER'].map({'YES': 1, 'NO': 0})

# Ajustar colunas que usam [1, 2] para [0, 1]
cols_to_fix = df_encoded.columns.drop(['AGE', 'LUNG_CANCER'])
for col in cols_to_fix:
    df_encoded[col] = df_encoded[col].apply(lambda x: x - 1)

# Renomear 'LUNG_CANCER' para 'TARGET' (opcional, só organização)
df_encoded.rename(columns={'LUNG_CANCER': 'TARGET'}, inplace=True)

# Exibir os dados tratados
print("Pré-processamento concluído. Amostra dos dados:")
print(df_encoded.head())


Pré-processamento concluído. Amostra dos dados:
   GENDER  AGE  SMOKING  YELLOW_FINGERS  ANXIETY  PEER_PRESSURE  \
0       0   69        0               1        1              0   
1       0   74        1               0        0              0   
2      -1   59        0               0        0              1   
3       0   63        1               1        1              0   
4      -1   63        0               1        0              0   

   CHRONIC DISEASE  FATIGUE   ALLERGY   WHEEZING  ALCOHOL CONSUMING  COUGHING  \
0                0         1         0         1                  1         1   
1                1         1         1         0                  0         0   
2                0         1         0         1                  0         1   
3                0         0         0         0                  1         0   
4                0         0         0         1                  0         1   

   SHORTNESS OF BREATH  SWALLOWING DIFFICULTY  CHEST PAIN  TAR

# Preparar os dados para o modelo

In [16]:
# Separar features e alvo
X = df_encoded.drop('TARGET', axis=1)
y = df_encoded['TARGET']

# Dividir em treino e teste (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Modelo 1 – Random Forest

In [17]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Avaliação
y_pred_rf = rf_model.predict(X_test)
print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred_rf))


=== Random Forest ===
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.98      0.98      0.98        60

    accuracy                           0.97        62
   macro avg       0.74      0.74      0.74        62
weighted avg       0.97      0.97      0.97        62

Matriz de Confusão:
[[ 1  1]
 [ 1 59]]


# Modelo 2 – Regressão Logística

In [18]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

# Avaliação
y_pred_log = log_model.predict(X_test)
print("\n=== Regressão Logística ===")
print(classification_report(y_test, y_pred_log))
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred_log))



=== Regressão Logística ===
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.98      0.98      0.98        60

    accuracy                           0.97        62
   macro avg       0.74      0.74      0.74        62
weighted avg       0.97      0.97      0.97        62

Matriz de Confusão:
[[ 1  1]
 [ 1 59]]


# Previsão de probabilidade para novos pacientes

**Função:** prever_cancer()

Ela recebe uma lista com as respostas de um novo paciente e retorna a probabilidade de câncer de pulmão com base no modelo treinado.

In [23]:
def prever_cancer(modelo, dados_paciente):

    prob = modelo.predict_proba([dados_paciente])[0][1]  # probabilidade de classe 1
    print(f"Probabilidade de câncer de pulmão: {prob * 100:.2f}%")


In [24]:
# lista dos nomes das perguntas ou variáveis que o modelo usa para fazer a previsão
print(X.columns.tolist())

['GENDER', 'AGE', 'SMOKING', 'YELLOW_FINGERS', 'ANXIETY', 'PEER_PRESSURE', 'CHRONIC DISEASE', 'FATIGUE ', 'ALLERGY ', 'WHEEZING', 'ALCOHOL CONSUMING', 'COUGHING', 'SHORTNESS OF BREATH', 'SWALLOWING DIFFICULTY', 'CHEST PAIN']


In [25]:
# Criar DataFrame com as colunas originais e 1 linha (novo paciente)
novo_paciente_df = pd.DataFrame([novo_paciente], columns=X.columns)

prob = rf_model.predict_proba(novo_paciente_df)[0][1]
print(f"Probabilidade: {prob*100:.2f}%")


Probabilidade: 99.00%


# Previsão Feita Manualmente

In [30]:
colunas = X.columns.tolist()  # lista com nomes das features


In [33]:
def prever_cancer(modelo, dados_paciente):
    """
    Recebe:
        - modelo: modelo treinado (ex: rf_model)
        - dados_paciente: lista com valores dos sintomas/variáveis

    Mostra as respostas do paciente com nomes das variáveis e calcula a probabilidade.
    """
    colunas = X.columns.tolist()

    # Mostrar respostas organizadas
    print("Respostas do paciente:")
    for nome, valor in zip(colunas, dados_paciente):
        print(f"  {nome}: {valor}")

    # Calcular probabilidade
    prob = modelo.predict_proba([dados_paciente])[0][1]
    print(f"\nProbabilidade de câncer de pulmão: {prob * 100:.2f}%")


### **Paciente 1**

In [34]:
novo_paciente = [1, 65, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1]
prever_cancer(rf_model, novo_paciente)


Respostas do paciente:
  GENDER: 1
  AGE: 65
  SMOKING: 1
  YELLOW_FINGERS: 1
  ANXIETY: 0
  PEER_PRESSURE: 1
  CHRONIC DISEASE: 0
  FATIGUE : 1
  ALLERGY : 0
  WHEEZING: 1
  ALCOHOL CONSUMING: 1
  COUGHING: 1
  SHORTNESS OF BREATH: 1
  SWALLOWING DIFFICULTY: 0
  CHEST PAIN: 1

Probabilidade de câncer de pulmão: 99.00%


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


### **Paciente 2**

In [36]:
# Exemplo paciente com poucos sintomas preocupantes
novo_paciente_baixo_risco = [
    0,    # GENDER: F (0)
    30,   # AGE: 30 anos (mais jovem)
    0,    # SMOKING: Não fuma
    0,    # YELLOW_FINGERS: Não
    0,    # ANXIETY: Não
    0,    # PEER_PRESSURE: Não
    0,    # CHRONIC DISEASE: Não
    0,    # FATIGUE: Não
    0,    # ALLERGY: Não
    0,    # WHEEZING: Não
    0,    # ALCOHOL CONSUMING: Não
    0,    # COUGHING: Não
    0,    # SHORTNESS OF BREATH: Não
    0,    # SWALLOWING DIFFICULTY: Não
    0     # CHEST PAIN: Não
]

prever_cancer(rf_model, novo_paciente_baixo_risco)


Respostas do paciente:
  GENDER: 0
  AGE: 30
  SMOKING: 0
  YELLOW_FINGERS: 0
  ANXIETY: 0
  PEER_PRESSURE: 0
  CHRONIC DISEASE: 0
  FATIGUE : 0
  ALLERGY : 0
  WHEEZING: 0
  ALCOHOL CONSUMING: 0
  COUGHING: 0
  SHORTNESS OF BREATH: 0
  SWALLOWING DIFFICULTY: 0
  CHEST PAIN: 0

Probabilidade de câncer de pulmão: 47.00%


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


# **Tradução**

In [50]:
# Limpar espaços nos nomes das colunas
df_encoded.columns = df_encoded.columns.str.strip()

# Dicionário de tradução com nomes limpos
traducao_sintomas = {
    'GENDER': 'Gênero (1=Masculino, 0=Feminino)',
    'AGE': 'Idade',
    'SMOKING': 'Fuma',
    'YELLOW_FINGERS': 'Dedos amarelados',
    'ANXIETY': 'Ansiedade',
    'PEER_PRESSURE': 'Pressão dos colegas',
    'CHRONIC DISEASE': 'Doença crônica',
    'FATIGUE': 'Fadiga',
    'ALLERGY': 'Alergia',
    'WHEEZING': 'Chiado no peito',
    'ALCOHOL CONSUMING': 'Consumo de álcool',
    'COUGHING': 'Tosse',
    'SHORTNESS OF BREATH': 'Falta de ar',
    'SWALLOWING DIFFICULTY': 'Dificuldade para engolir',
    'CHEST PAIN': 'Dor no peito'
}


In [51]:
def prever_cancer_traduzido(modelo, dados_paciente):
    colunas = df_encoded.columns.drop('TARGET').tolist()

    print("Respostas do paciente:")
    for nome, valor in zip(colunas, dados_paciente):
        texto = traducao_sintomas.get(nome, nome)
        print(f"  {texto}: {valor}")

    prob = modelo.predict_proba([dados_paciente])[0][1]
    print(f"\nProbabilidade de câncer de pulmão: {prob * 100:.2f}%")

### **Paciente 3**

In [52]:
paciente_intermediario = [
    1,    # GÊNERO: Masculino
    50,   # IDADE: 50 anos
    1,    # Fuma
    0,    # Dedos amarelados: Não
    1,    # Ansiedade: Sim
    0,    # Pressão dos colegas: Não
    1,    # Doença crônica: Sim
    0,    # Fadiga: Não
    0,    # Alergia: Não
    1,    # Chiado no peito: Sim
    1,    # Consumo de álcool: Sim
    0,    # Tosse: Não
    1,    # Falta de ar: Sim
    0,    # Dificuldade para engolir: Não
    0     # Dor no peito: Não
]

In [53]:
prever_cancer_traduzido(rf_model, paciente_intermediario)


Respostas do paciente:
  Gênero (1=Masculino, 0=Feminino): 1
  Idade: 50
  Fuma: 1
  Dedos amarelados: 0
  Ansiedade: 1
  Pressão dos colegas: 0
  Doença crônica: 1
  Fadiga: 0
  Alergia: 0
  Chiado no peito: 1
  Consumo de álcool: 1
  Tosse: 0
  Falta de ar: 1
  Dificuldade para engolir: 0
  Dor no peito: 0

Probabilidade de câncer de pulmão: 73.00%


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


# Projeto: Previsão de Câncer de Pulmão com Machine Learning

Este notebook apresenta um projeto de Machine Learning para prever a probabilidade de câncer de pulmão em pacientes, utilizando dados clínicos e sintomas.

---

## Dataset

Os dados utilizados são do dataset público disponível no Kaggle:  
[Lung Cancer Dataset - Kaggle](https://www.kaggle.com/datasets/mysarahmadbhat/lung-cancer)

Este conjunto contém informações como idade, gênero, hábitos de vida (tabagismo, consumo de álcool), sintomas clínicos e diagnóstico confirmado.

---

## Objetivo

Desenvolver e avaliar modelos preditivos (Random Forest e Regressão Logística) para classificar pacientes com e sem câncer de pulmão, com base nos sintomas e histórico.

---

## Metodologia

- **Carregamento e exploração dos dados**  
- **Pré-processamento:** limpeza, codificação e tratamento de dados faltantes  
- **Treinamento dos modelos de classificação**  
- **Avaliação dos modelos usando métricas de desempenho**  
- **Função para previsão individual baseada nos sintomas do paciente**

---

## Tecnologias Utilizadas

- Python (pandas, scikit-learn)  
- Google Colab (ambiente de desenvolvimento)  
- Dataset público do Kaggle  

---

## Referências

- Dataset: https://www.kaggle.com/datasets/mysarahmadbhat/lung-cancer  
- Documentação scikit-learn: https://scikit-learn.org/stable/

---

**Nota:** Este projeto é educacional e não substitui diagnóstico médico.
